In [1]:
# ==========================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==========================================================

# PyTorch
import torch

# Dataset y DataLoader
from torch.utils.data import Dataset, DataLoader

# Manipulación de datos
import numpy as np
import pandas as pd

# Gráficos
from matplotlib import pyplot as plt

# Barra de progreso
from tqdm import tqdm

# Métricas
from sklearn.metrics import confusion_matrix, accuracy_score

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
# ==========================================================
# 2. DETECTAR GPU
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo utilizado:", device)

Dispositivo utilizado: cpu


In [4]:
# ==========================================================
# 3. CARGAR DATASET
# ==========================================================

data = pd.read_csv("mening missing 12.csv")

print("Dataset cargado correctamente.")

print("Dimensiones del dataset:")
print(data.shape)

display(data.head())

Dataset cargado correctamente.
Dimensiones del dataset:
(1200, 14)


,Patient_ID,Age,Gender,WBC_Count,Protein_Level,Glucose_Level,Pathogen_Present,Diagnosis,Outcome,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level,Risk_Level
0,1,101.0,Female,8624.0,16.0,83.0,No,Viral,Recovered,15.0,7269.0,160949.0,71.0,Moderate Risk
1,2,78.0,Male,22623.0,200.0,41.0,No,Unknown,Recovered,18.0,6532.0,371741.0,41.0,High Risk
2,3,8.0,Female,12908.0,39.0,3.0,No,Unknown,Recovered,16.0,7417.0,180403.0,22.0,Moderate Risk
3,4,104.0,Female,15072.0,58.0,36.0,Yes,Bacterial,Recovered,7.0,13792.0,132254.0,48.0,Moderate Risk
4,5,38.0,Female,18623.0,152.0,34.0,Yes,Bacterial,Recovered,5.0,17054.0,134941.0,28.0,High Risk


In [5]:
# ==========================================================
# 4. EXPLORACIÓN DEL DATASET
# ==========================================================

print("Columnas del dataset:")
print(list(data.columns))

print("\nInformación general:")
print(data.info())

print("\nValores nulos por columna:")
print(data.isnull().sum())

Columnas del dataset:
['Patient_ID', 'Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Pathogen_Present', 'Diagnosis', 'Outcome', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level', 'Risk_Level']

Información general:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Patient_ID        1200 non-null   int64  
 1   Age               1193 non-null   float64
 2   Gender            1191 non-null   str    
 3   WBC_Count         1192 non-null   float64
 4   Protein_Level     1190 non-null   float64
 5   Glucose_Level     1192 non-null   float64
 6   Pathogen_Present  1192 non-null   str    
 7   Diagnosis         1188 non-null   str    
 8   Outcome           1190 non-null   str    
 9   Hemoglobin        1181 non-null   float64
 10  WBC_Blood_Count   1190 non-null   float64
 11  Platelets         1188 non-null   float64
 12  C

In [6]:
# ==========================================================
# 5. ESTADÍSTICAS BÁSICAS
# ==========================================================

display(data.describe())

,Patient_ID,Age,WBC_Count,Protein_Level,Glucose_Level,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level
count,1200.000000,1193.000000,1192.000000,1190.000000,1192.000000,1181.000000,1190.000000,1188.000000,1187.000000
mean,600.500000,47.432523,12130.966443,109.463866,52.633389,9.998307,11295.922689,201626.741582,27.497051
std,346.554469,24.571305,5670.489634,72.122291,34.695494,5.577193,4956.524846,90936.534040,26.373065
min,1.000000,0.000000,2017.000000,0.000000,0.000000,1.000000,4013.000000,100204.000000,0.000000
25%,300.750000,29.000000,7007.250000,43.000000,26.000000,5.000000,6746.750000,125491.250000,3.000000
50%,600.500000,43.000000,12282.000000,108.000000,53.000000,12.000000,9782.500000,161890.500000,25.000000
75%,900.250000,61.000000,16895.750000,167.000000,66.000000,15.000000,15806.250000,275962.250000,42.000000
max,1200.000000,119.000000,24947.000000,297.000000,149.000000,18.000000,19988.000000,399110.000000,99.000000


In [9]:
# ==========================================================
# 6. LIMPIEZA DE DATOS (CORREGIDA PARA PANDAS NUEVO)
# ==========================================================

# Convertir valores "?" en NaN
data = data.replace("?", np.nan)

# Intentar convertir todas las columnas a números
# Los valores que no puedan convertirse se vuelven NaN
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Imputar valores nulos
for col in data.columns:

    # si la columna sigue teniendo tipo objeto → categórica
    if data[col].dtype == "object":
        data[col] = data[col].fillna(data[col].mode()[0])

    # si es numérica
    else:
        data[col] = data[col].fillna(data[col].mean())

print("Valores nulos imputados correctamente.")

print("\nNulos restantes por columna:")
print(data.isnull().sum())

Valores nulos imputados correctamente.

Nulos restantes por columna:
Patient_ID             0
Age                    0
Gender              1200
WBC_Count              0
Protein_Level          0
Glucose_Level          0
Pathogen_Present    1200
Diagnosis           1200
Outcome             1200
Hemoglobin             0
WBC_Blood_Count        0
Platelets              0
CRP_Level              0
Risk_Level          1200
dtype: int64


In [11]:
# ==========================================================
# 7. VERIFICAR DATASET DESPUÉS DE LIMPIEZA
# ==========================================================

print("Información del dataset después de limpieza:")
print(data.info())

print("\nPrimeras filas:")
display(data.head())

print("\nValores nulos restantes:")
print(data.isnull().sum())

Información del dataset después de limpieza:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Patient_ID        1200 non-null   int64  
 1   Age               1200 non-null   float64
 2   Gender            0 non-null      float64
 3   WBC_Count         1200 non-null   float64
 4   Protein_Level     1200 non-null   float64
 5   Glucose_Level     1200 non-null   float64
 6   Pathogen_Present  0 non-null      float64
 7   Diagnosis         0 non-null      float64
 8   Outcome           0 non-null      float64
 9   Hemoglobin        1200 non-null   float64
 10  WBC_Blood_Count   1200 non-null   float64
 11  Platelets         1200 non-null   float64
 12  CRP_Level         1200 non-null   float64
 13  Risk_Level        0 non-null      float64
dtypes: float64(13), int64(1)
memory usage: 131.4 KB
None

Primeras filas:


,Patient_ID,Age,Gender,WBC_Count,Protein_Level,Glucose_Level,Pathogen_Present,Diagnosis,Outcome,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level,Risk_Level
0,1,101.0,NaN,8624.0,16.0,83.0,NaN,NaN,NaN,15.0,7269.0,160949.0,71.0,NaN
1,2,78.0,NaN,22623.0,200.0,41.0,NaN,NaN,NaN,18.0,6532.0,371741.0,41.0,NaN
2,3,8.0,NaN,12908.0,39.0,3.0,NaN,NaN,NaN,16.0,7417.0,180403.0,22.0,NaN
3,4,104.0,NaN,15072.0,58.0,36.0,NaN,NaN,NaN,7.0,13792.0,132254.0,48.0,NaN
4,5,38.0,NaN,18623.0,152.0,34.0,NaN,NaN,NaN,5.0,17054.0,134941.0,28.0,NaN



Valores nulos restantes:
Patient_ID             0
Age                    0
Gender              1200
WBC_Count              0
Protein_Level          0
Glucose_Level          0
Pathogen_Present    1200
Diagnosis           1200
Outcome             1200
Hemoglobin             0
WBC_Blood_Count        0
Platelets              0
CRP_Level              0
Risk_Level          1200
dtype: int64


In [12]:
# ==========================================================
# 8. SEPARAR FEATURES Y TARGET
# ==========================================================

# última columna será la variable objetivo
target = data.columns[-1]

print("Variable objetivo:", target)

# features = todas menos la última
features = list(data.columns[:-1])

print("Número de features:", len(features))
print("Features utilizadas:")
print(features)

# crear arrays
X = data[features].values
y = data[target].values

print("\nForma de X:", X.shape)
print("Forma de y:", y.shape)

Variable objetivo: Risk_Level
Número de features: 13
Features utilizadas:
['Patient_ID', 'Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Pathogen_Present', 'Diagnosis', 'Outcome', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level']

Forma de X: (1200, 13)
Forma de y: (1200,)


In [14]:
# ==========================================================
# 10. NORMALIZACIÓN DE DATOS
# ==========================================================

def normalizar(X):

    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)

    X_norm = (X - mu) / sigma

    return X_norm, mu, sigma


X_norm, mu, sigma = normalizar(X)

print("Datos normalizados correctamente.")

print("Media de cada feature:")
print(np.round(mu,3))

print("Desviación estándar:")
print(np.round(sigma,3))

Datos normalizados correctamente.
Media de cada feature:
[6.00500000e+02 4.74330000e+01            nan 1.21309660e+04
 1.09464000e+02 5.26330000e+01            nan            nan
            nan 9.99800000e+00 1.12959230e+04 2.01626742e+05
 2.74970000e+01]
Desviación estándar:
[3.464100e+02 2.448900e+01          nan 5.649185e+03 7.179100e+01
 3.456500e+01          nan          nan          nan 5.531000e+00
 4.933755e+03 9.044262e+04 2.621900e+01]


In [15]:
# ==========================================================
# 11. DIVISIÓN TRAIN / TEST
# ==========================================================

split = int(len(X_norm) * 0.8)

X_train = X_norm[:split]
y_train = y[:split]

X_test = X_norm[split:]
y_test = y[split:]

print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

Tamaño entrenamiento: (960, 13)
Tamaño prueba: (240, 13)


In [16]:
# ==========================================================
# 13. VERIFICAR DIMENSIONES
# ==========================================================

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (960, 13)
y_train: (960,)
X_test: (240, 13)
y_test: (240,)
